# Local manga OCR — ncl_sae

Uses the existing CUDA PyTorch in `ncl_sae` on the RTX 5060 Laptop GPU (8 GB).
Open this notebook in VS Code or Jupyter and select that environment as the kernel.
Run the installation cell once, **restart the kernel**, then run the remaining cells.
The install cell does not replace PyTorch or torchvision. Internet is needed initially
for packages and model downloads. The supplied dataset path is already set.

**Two hours is a target, not a measured guarantee.** Three training sequences are timed
before the full run. BF16 autocast, TF32, compiled edit distance, and a smaller validation
grid reduce work; all 80 development and all 15 test sequences are still processed.
OCR generation keeps its 128-token limit and page images keep their original resolution.
The baseline and adapted model share the same extraction settings for a fair comparison.

Magi's detector, OCR, embeddings, and local associations are frozen. The story-text and
narration classifiers are trained from development labels; anonymous identities are
matched across each triplet. No hosted inference APIs are used.

Magi has restricted research/non-commercial terms. Confirm the task allows that model.
Sources: https://huggingface.co/ragavsachdeva/magiv2 and https://github.com/ragavsachdeva/magi.
This notebook was prepared with AI assistance; write your submission README from your
own experiments. No end-to-end speed or accuracy result is claimed in advance.


## Reproducing the tuning experiment
This version evaluates 96 prediction configurations and keeps the official metric
definitions unchanged. The selected result is re-evaluated using a freshly imported,
unmodified copy of the supplied score.py. The previous validation result was 0.4037
balanced joint F1; reruns may differ with model, precision, or environment changes.
This is repeated tuning on the same validation set, not a new independent test.

By default, original image caches are reused from manga_local_run when their signatures
match. New results go to manga_tuned_notebook_run. All training, tuning and test-export
code is contained in this notebook; tune_cached.py is not required.


## Install missing notebook dependencies
Run once in `ncl_sae`, then restart the kernel. This changes packages in that environment,
including pinning transformers and scikit-learn for this notebook, but keeps its GPU
PyTorch/torchvision installation. If you need to preserve other projects' dependencies,
clone the environment first. Subsequent runs can skip this cell.

In [ ]:
%pip install -q transformers==4.44.2 timm==1.0.9 einops==0.8.0 pulp==2.9.0 sentencepiece==0.2.0 rapidfuzz==3.9.7 scikit-learn==1.5.2 joblib==1.4.2
%pip install -q "numpy<2" pandas scipy pillow tqdm ipywidgets nbformat

In [ ]:
from pathlib import Path
import json, re, random, hashlib, time, sys, importlib.util, subprocess
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm.auto import tqdm
from rapidfuzz.fuzz import ratio
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT = Path(r"C:/Users/Raaghav/Desktop/maanga/BYTE ML Task Data")
WORK = DATA_ROOT.parent / 'manga_tuned_notebook_run'
PREVIOUS_RUN = DATA_ROOT.parent / 'manga_local_run'
WORK.mkdir(parents=True, exist_ok=True)
RUN_STARTED = time.perf_counter()
TARGET_SECONDS = 2 * 60 * 60
USE_AMP = True  # BF16 reduces activation memory; set False if it causes numerical errors.
OCR_BATCH = 8
EMBED_BATCH = 8
print('Python:', sys.executable)
if 'ncl_sae' not in sys.executable.lower():
    raise RuntimeError('Select the ncl_sae interpreter/kernel, then rerun from the start.')
if not torch.cuda.is_available():
    raise RuntimeError('This kernel cannot use CUDA. Select ncl_sae, not the system Python.')
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
assert not USE_AMP or torch.cuda.is_bf16_supported(), 'Set USE_AMP=False on this GPU.'
probe = torch.randn(128, 128, device='cuda')
print('CUDA check:', float((probe @ probe).mean()))
del probe

manifest = json.loads((DATA_ROOT / 'sequences.json').read_text())
seq_by_id = {s['sequence_id']: s for s in manifest}
assert len(seq_by_id) == len(manifest), 'Duplicate sequence IDs'
for s in manifest:
    assert len(s['images']) == 3
    for p in s['images']:
        assert (DATA_ROOT / p).is_file(), p

spec = importlib.util.spec_from_file_location('task_score', DATA_ROOT / 'score.py')
scorer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scorer)
gold = scorer.load_jsonl(DATA_ROOT / 'development/labels.jsonl', reference=True)
dev_ids = sorted(s['sequence_id'] for s in manifest if s['split'] == 'development')
test_ids = [s['sequence_id'] for s in manifest if s['split'] == 'test']
assert set(gold) == set(dev_ids)
assert len(dev_ids) == 80 and len(test_ids) == 15
assert torch.cuda.is_available(), 'CUDA is required for this run.'
print(DATA_ROOT, torch.cuda.get_device_name(0))
print(f'{len(dev_ids)} development sequences; {len(test_ids)} test sequences')
# Load an untouched scorer for final verification of every selected metric.
official_spec = importlib.util.spec_from_file_location('official_score', DATA_ROOT / 'score.py')
official_score = importlib.util.module_from_spec(official_spec)
official_spec.loader.exec_module(official_score)


## Split whole sequences
Exact duplicate pages are grouped so that they cannot cross the split. This does not
detect neighboring pages from the same book. If you know the book/chapter groups,
populate `BOOK_GROUPS` before running this cell for a stronger holdout.

In [ ]:
BOOK_GROUPS = {}  # Optional: sequence_id -> book/chapter group.
parent = {sid: sid for sid in dev_ids}
def root(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def unite(a, b):
    parent[root(b)] = root(a)
seen = {}
for sid in dev_ids:
    for rel in seq_by_id[sid]['images']:
        digest = hashlib.sha256((DATA_ROOT / rel).read_bytes()).hexdigest()
        if digest in seen:
            unite(sid, seen[digest])
        seen[digest] = sid
book_seen = {}
for sid, book in BOOK_GROUPS.items():
    assert sid in parent
    if book in book_seen:
        unite(sid, book_seen[book])
    book_seen[book] = sid
groups = [root(sid) for sid in dev_ids]
tr, va = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
              .split(dev_ids, groups=groups))
train_ids = [dev_ids[i] for i in tr]
val_ids = [dev_ids[i] for i in va]
assert not {groups[i] for i in tr} & {groups[i] for i in va}
(WORK / 'split.json').write_text(json.dumps({'train': train_ids, 'validation': val_ids}, indent=2))
print('Train:', len(train_ids), 'Validation:', len(val_ids))

## Frozen detector and OCR
Magi already orders its text boxes using panels. Keep that order. An empty character
bank makes its naming API return `Other`, so we use crop embeddings and local cluster
IDs instead. No real names or reference speaker labels enter inference.

The resolved model revision is saved. Keep `run_metadata.json` to reproduce the run;
put its revision in `MODEL_REVISION` when rerunning elsewhere. Loading this model
executes its custom Python code (`trust_remote_code=True`).

In [ ]:
from huggingface_hub import model_info
from transformers import AutoModel

MODEL_ID = 'ragavsachdeva/magiv2'
MODEL_REVISION = None  # Use the saved commit SHA to reproduce an earlier run.
prior = WORK / 'run_metadata.json'
prior = prior if prior.exists() else PREVIOUS_RUN / 'run_metadata.json'
saved_revision = json.loads(prior.read_text()).get('revision') if prior.exists() else None
revision = MODEL_REVISION or saved_revision or model_info(MODEL_ID).sha
model = AutoModel.from_pretrained(MODEL_ID, revision=revision,
                                  trust_remote_code=True).cuda().eval()
metadata = {'model': MODEL_ID, 'revision': revision, 'seed': SEED,
            'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
            'extractor_version': 2, 'ocr_max_new_tokens': 128,
            'amp': USE_AMP, 'ocr_batch': OCR_BATCH, 'embed_batch': EMBED_BATCH}
(WORK / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
(WORK / 'environment.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))
CACHE = PREVIOUS_RUN / ('cache_' + revision[:12])
CACHE.mkdir(parents=True, exist_ok=True)

def read_image(rel):
    with Image.open(DATA_ROOT / rel) as im:
        return np.array(im.convert('L').convert('RGB'))

def extract_sequence(sid):
    paths = seq_by_id[sid]['images']
    fingerprint = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode())
    for rel in paths:
        fingerprint.update((DATA_ROOT / rel).read_bytes())
    signature = fingerprint.hexdigest()
    destination = CACHE / f'{sid}.joblib'
    if destination.exists():
        cached = joblib.load(destination)
        if cached['signature'] == signature:
            return cached['pages']
    pages = []
    torch.cuda.synchronize()
    started = time.time()
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16, enabled=USE_AMP):
        for rel in paths:
            image = read_image(rel)
            result = model.predict_detections_and_associations([image])[0]
            boxes = result['texts']
            texts = model.predict_ocr([image], [boxes], batch_size=OCR_BATCH,
                                      max_new_tokens=128)[0]
            # Visual text features let the small filter learn beyond word matching.
            text_emb = model.predict_crop_embeddings([image], [boxes], batch_size=EMBED_BATCH)[0]
            char_emb = model.predict_crop_embeddings(
                [image], [result['characters']], batch_size=EMBED_BATCH)[0]
            assert len(texts) == len(boxes) == len(result['is_essential_text'])
            result.update(ocr=texts, size=[image.shape[1], image.shape[0]],
                          text_embeddings=text_emb.float().cpu().numpy(),
                          char_embeddings=char_emb.float().cpu().numpy())
            pages.append(result)
    torch.cuda.synchronize()
    tmp = destination.with_suffix('.tmp')
    joblib.dump({'signature': signature, 'pages': pages}, tmp)
    tmp.replace(destination)
    with (WORK / 'extraction_log.jsonl').open('a') as f:
        f.write(json.dumps({'sequence_id': sid, 'seconds': time.time()-started,
                            'regions': sum(len(p['ocr']) for p in pages)}) + '\n')
    return pages

# Time three training sequences; the first includes CUDA warm-up.
smoke_ids = train_ids[:3]
smoke_seconds = []
for smoke_id in smoke_ids:
    t0 = time.perf_counter()
    smoke = extract_sequence(smoke_id)
    torch.cuda.synchronize()
    seconds = time.perf_counter() - t0
    smoke_seconds.append(seconds)
    print(f'{smoke_id}: {seconds:.1f}s; regions: {[len(p["ocr"]) for p in smoke]}')
display(pd.DataFrame({'text': smoke[0]['ocr'], 'essential': smoke[0]['is_essential_text']}))

logs = []
for log_path in [PREVIOUS_RUN / 'extraction_log.jsonl', WORK / 'extraction_log.jsonl']:
    if log_path.exists():
        logs.extend(json.loads(line) for line in log_path.read_text().splitlines() if line.strip())
recent = {r['sequence_id']: r['seconds'] for r in logs}
sample = [recent[sid] for sid in smoke_ids]
remaining = sum(not (CACHE / f'{sid}.joblib').exists() for sid in dev_ids + test_ids)
elapsed = time.perf_counter() - RUN_STARTED
estimate = elapsed + float(np.median(sample)) * remaining + 15 * 60
conservative = elapsed + max(sample) * remaining * 1.3 + 20 * 60
print(f'Rough total: {estimate/60:.0f} min; conservative estimate: {conservative/60:.0f} min')
print('Includes elapsed setup plus a scoring allowance; page complexity and download speeds vary.')
if conservative > TARGET_SECONDS:
    print('WARNING: the two-hour target is not supported by this timing. No pages will be skipped.')
(WORK / 'timing_estimate.json').write_text(json.dumps({
    'sample_seconds': sample, 'remaining_sequences': remaining,
    'estimated_total_minutes': estimate/60, 'conservative_minutes': conservative/60}, indent=2))


If the smoke test fails, fix the environment before running the full cache. A crash
does not create a fake empty prediction. Completed sequences can be reused on rerun.

In [ ]:
dev_cache = {sid: extract_sequence(sid) for sid in tqdm(dev_ids, desc='Development OCR')}

## Weak labels from the training transcripts
There are no balloon boxes in the references. Align OCR regions to short spans of
reference text. Confident matches teach inclusion and narration. Very dissimilar,
nonessential regions on pages with good reference coverage provide weak negatives.
Everything ambiguous is ignored. These negatives can still be wrong; the audit table
below is part of the experiment, not ground truth.

This first version does not solve all split/merged-balloon cases. Short punctuation
is accepted only on an exact match. Validation labels are never used to fit either head.

In [ ]:
def clean(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

def reference_candidates(rows):
    candidates = []
    for start in range(len(rows)):
        for width in range(1, 4):
            chunk = rows[start:start+width]
            if len(chunk) != width:
                continue
            speakers = {r['speaker'] for r in chunk}
            if width > 1 and len(speakers) != 1:
                continue
            candidates.append((scorer.norm(' '.join(r['text'] for r in chunk)),
                               chunk[0]['speaker'], set(range(start, start+width))))
    return candidates

def match_page(page, rows):
    candidates = reference_candidates(rows)
    matches = []
    covered = set()
    for text in page['ocr']:
        query = scorer.norm(text)
        ranked = sorted([(ratio(query, target)/100, speaker, indices, target)
                         for target, speaker, indices in candidates],
                        key=lambda x: x[0], reverse=True)
        best = ranked[0] if ranked else (0., None, set(), '')
        confidence = best[0] >= 0.90 and (len(query) >= 4 or query == best[3])
        # Repeated wording with different identities is unsafe narration supervision.
        ambiguous = any(r[1] != best[1] and r[0] >= best[0]-0.03 for r in ranked[1:])
        matches.append((best, confidence, ambiguous))
        if confidence:
            covered |= best[2]
    return matches, len(covered)/max(1, len(rows)) if rows else 1.

def region_features(page, i):
    w, h = page['size']
    x1, y1, x2, y2 = page['texts'][i]
    text = clean(page['ocr'][i])
    n = max(len(text), 1)
    associated = {int(t) for t, c in page['text_character_associations']}
    tails = {int(t) for t, c in page.get('text_tail_associations', [])}
    geometry = [x1/w, y1/h, x2/w, y2/h, (x2-x1)/w, (y2-y1)/h,
                np.log1p(len(text)), len(text.split())/50,
                sum(c.isalpha() for c in text)/n,
                sum(c.isdigit() for c in text)/n,
                sum(c in '.!?' for c in text)/n,
                float(page['is_essential_text'][i]), float(i in associated), float(i in tails)]
    emb = np.asarray(page['text_embeddings'][i], dtype=np.float32)
    emb = emb / max(float(np.linalg.norm(emb)), 1e-8)
    return np.concatenate([geometry, emb]).astype(np.float32)

def training_examples(ids):
    X, y, nx, ny, audit = [], [], [], [], []
    for sid in ids:
        for p, (page, rows) in enumerate(zip(dev_cache[sid], gold[sid])):
            matches, coverage = match_page(page, rows)
            for i, (best, confident, ambiguous) in enumerate(matches):
                if not clean(page['ocr'][i]):
                    continue
                label = None
                if confident:
                    label = 1
                elif best[0] < 0.25 and coverage >= 0.8 and not page['is_essential_text'][i]:
                    label = 0
                if label is not None:
                    features = region_features(page, i)
                    X.append(features); y.append(label)
                    audit.append({'sequence_id': sid, 'page': p+1, 'region': i,
                                  'ocr': page['ocr'][i], 'weak_include': label,
                                  'similarity': best[0], 'reference_coverage': coverage})
                    if confident and not ambiguous and best[1] is not None:
                        nx.append(features); ny.append(int(best[1] == 'NARRATION'))
    return np.asarray(X), np.asarray(y), np.asarray(nx), np.asarray(ny), pd.DataFrame(audit)

def fit_heads(ids):
    X, y, nx, ny, audit = training_examples(ids)
    if len(set(y)) < 2 or len(set(ny)) < 2:
        raise RuntimeError('Insufficient weak classes. Inspect training alignment; do not fabricate labels.')
    def fit(x, target):
        learner = make_pipeline(StandardScaler(), LogisticRegression(
            C=0.05, class_weight='balanced', max_iter=2000, random_state=SEED))
        return learner.fit(x, target)
    return {'story': fit(X, y), 'narration': fit(nx, ny)}, audit, {
        'story_counts': dict(Counter(map(int, y))), 'narration_counts': dict(Counter(map(int, ny)))}

heads, audit, counts = fit_heads(train_ids)
audit.to_csv(WORK / 'training_alignment_audit.csv', index=False)
(WORK / 'training_counts.json').write_text(json.dumps(counts, indent=2))
joblib.dump(heads, WORK / 'validation_heads.joblib')
print(counts)
display(audit.sample(min(15, len(audit)), random_state=SEED))

## Anonymous identities across pages
Start with Magi's within-page character clusters. Merge clusters from different pages
only when every pair in the merged group passes the similarity threshold. Never merge
two different clusters from the same page. This conservative rule avoids chains of weak
matches, but cannot repair a bad within-page cluster.

Unassociated lines get separate unresolved IDs. That loses continuity for off-panel
speakers, but avoids pretending all off-panel speech comes from one person.

In [ ]:
def unit(x):
    x = np.asarray(x, dtype=np.float32)
    return x / max(float(np.linalg.norm(x)), 1e-8)

def character_ids(pages, threshold):
    nodes, appearance = [], {}
    for p, page in enumerate(pages):
        clusters = defaultdict(list)
        for c, label in enumerate(page['character_cluster_labels']):
            clusters[int(label)].append(c)
        for members in clusters.values():
            index = len(nodes)
            emb = unit(np.mean([unit(page['char_embeddings'][c]) for c in members], axis=0))
            nodes.append((p, emb))
            for c in members:
                appearance[p, c] = index
    groups = [{i} for i in range(len(nodes))]
    edges = sorted([(float(a[1] @ b[1]), i, j)
                    for i, a in enumerate(nodes) for j, b in enumerate(nodes[:i])
                    if a[0] != b[0]], reverse=True)
    for similarity, i, j in edges:
        if similarity < threshold:
            break
        gi = next(g for g in groups if i in g)
        gj = next(g for g in groups if j in g)
        if gi is gj:
            continue
        if {nodes[k][0] for k in gi} & {nodes[k][0] for k in gj}:
            continue
        if min(float(nodes[a][1] @ nodes[b][1]) for a in gi for b in gj) < threshold:
            continue
        merged = gi | gj
        groups = [g for g in groups if g is not gi and g is not gj] + [merged]
    mapping = {node: group for group, members in enumerate(sorted(groups, key=min)) for node in members}
    return {key: mapping[value] for key, value in appearance.items()}

def predict_pages(pages, heads=None, story_threshold=0.5, narration_threshold=0.7,
                  cluster_threshold=0.8):
    identities = character_ids(pages, cluster_threshold)
    names, output = {}, []
    for p, page in enumerate(pages):
        links = {int(t): int(c) for t, c in page['text_character_associations']}
        rows = []
        if heads and page['ocr']:
            features = np.stack([region_features(page, i) for i in range(len(page['ocr']))])
            keep_prob = heads['story'].predict_proba(features)[:, 1]
            narr_prob = heads['narration'].predict_proba(features)[:, 1]
        for i, raw in enumerate(page['ocr']):
            text = clean(raw)
            keep = keep_prob[i] >= story_threshold if heads and story_threshold is not None else page['is_essential_text'][i]
            if not text or not keep:
                continue
            if heads and narration_threshold is not None and narr_prob[i] >= narration_threshold:
                speaker = 'NARRATION'
            else:
                c = links.get(i)
                identity = ('character', identities[p, c]) if c is not None else ('unresolved', p, i)
                if identity not in names:
                    names[identity] = f'char{len(names)+1}'
                speaker = names[identity]
            rows.append({'speaker': speaker, 'text': text})
        output.append(rows)
    return output

def write_jsonl(path, predictions):
    with Path(path).open('w', encoding='utf-8') as f:
        for sid, pages in predictions.items():
            f.write(json.dumps({'sequence_id': sid, 'pages': pages}, ensure_ascii=False) + '\n')

def evaluate_config(name, config, active_heads):
    predictions = {sid: predict_pages(dev_cache[sid], active_heads, **config) for sid in val_ids}
    report = scorer.evaluate({sid: gold[sid] for sid in val_ids}, predictions)
    write_jsonl(WORK / f'{name}.jsonl', predictions)
    (WORK / f'{name}_scores.json').write_text(json.dumps(report, indent=2))
    return {'experiment': name, **config, **report['macro']}

## Separate the components and select on validation
Test frozen versus trained story filtering, narration off versus three thresholds,
and six cross-page merge thresholds. That makes 96 configurations. Select the largest
balanced joint F1 among candidates using at least one trained head and retaining a
text-order score of at least 0.8984. Record all results, including frozen candidates.
This rule prioritizes speaker quality without sacrificing more than 0.01 text score
relative to the original frozen baseline. It does not guarantee a test improvement.

In [ ]:
from functools import lru_cache
from rapidfuzz.distance import Levenshtein

# Computation shortcuts only: same distance and same text alignment/tie-breaking.
# They are removed before official verification below.
original_align = scorer.align_tokens
@lru_cache(maxsize=2048)
def alignment_indices(a, b):
    return original_align([(t, i) for i, t in enumerate(a)], [(t, i) for i, t in enumerate(b)])
def cached_align(a, b):
    return [(a[x[1]], b[y[1]], weight) for x, y, weight in alignment_indices(
        tuple(t for t, _ in a), tuple(t for t, _ in b))]
scorer.distance = Levenshtein.distance
scorer.align_tokens = cached_align

# Reuse model probabilities across thresholds rather than running classifiers 96 times.
class MemoHead:
    def __init__(self, learner):
        self.learner, self.memo = learner, {}
    def predict_proba(self, features):
        key = (features.shape, features.tobytes())
        if key not in self.memo:
            self.memo[key] = self.learner.predict_proba(features)
        return self.memo[key]
tuning_heads = {name: MemoHead(head) for name, head in heads.items()}
refs = {sid: gold[sid] for sid in val_ids}
results, configs = [], []
for keep in (None, 0.20, 0.35, 0.50):
    for narr in (None, 0.70, 0.85, 0.95):
        for cluster in (0.60, 0.70, 0.80, 0.85, 0.90, 0.95):
            configs.append({'story_threshold': keep, 'narration_threshold': narr,
                            'cluster_threshold': cluster})
for i, cfg in enumerate(tqdm(configs, desc='Validation configurations')):
    pred = {sid: predict_pages(dev_cache[sid], tuning_heads, **cfg) for sid in val_ids}
    report = scorer.evaluate(refs, pred)
    results.append({'experiment': f'tuned_{i:02d}', **cfg,
                    'uses_learned_head': cfg['story_threshold'] is not None or cfg['narration_threshold'] is not None,
                    **report['macro']})
comparison = pd.DataFrame(results)
comparison.to_csv(WORK / 'validation_grid.csv', index=False)
eligible = comparison[comparison.uses_learned_head & (comparison.text_order_score >= 0.8984)]
if eligible.empty:
    raise RuntimeError('No learned configuration meets the text constraint. Inspect the validation errors.')
best = eligible.sort_values(['balanced_joint_f1', 'text_order_score'], ascending=False).iloc[0]
selected = configs[int(best.experiment.split('_')[1])]
(WORK / 'selected_config.json').write_text(json.dumps(selected, indent=2))
predictions = {sid: predict_pages(dev_cache[sid], heads, **selected) for sid in val_ids}
write_jsonl(WORK / f'{best.experiment}.jsonl', predictions)

# Read the written predictions with the supplied parser and score them without patches.
written = official_score.load_jsonl(WORK / f'{best.experiment}.jsonl')
official_report = official_score.evaluate(refs, written)
for metric, value in official_report['macro'].items():
    assert np.isclose(value, float(best[metric]), atol=1e-10), metric
(WORK / f'{best.experiment}_scores.json').write_text(json.dumps(official_report, indent=2))
write_jsonl(WORK / 'validation_references.jsonl', refs)
# Reference loader converts UNKNOWN to None; the official reference loader accepts that.

baseline_rows = []
for name, active_heads, cfg in [
    ('frozen_page_local', None, {'cluster_threshold': 1.01}),
    ('frozen_sequence', None, {'cluster_threshold': 0.8}),
    ('original_adapted', heads, {'story_threshold': 0.35, 'narration_threshold': 0.7, 'cluster_threshold': 0.8})]:
    pred = {sid: predict_pages(dev_cache[sid], active_heads, **cfg) for sid in val_ids}
    report = official_score.evaluate(refs, pred)
    baseline_rows.append({'experiment': name, **report['macro']})
    write_jsonl(WORK / f'{name}.jsonl', pred)
    (WORK / f'{name}_scores.json').write_text(json.dumps(report, indent=2))
baseline_rows.append({'experiment': 'selected_tuned', **official_report['macro']})
pd.DataFrame(baseline_rows).to_csv(WORK / 'experiments.csv', index=False)
display(pd.DataFrame(baseline_rows))
print('Selected:', selected)
print('Every selected metric agrees with the unchanged official scorer.')
print('Validation was reused for tuning; this is not an independent test score.')
scorer.align_tokens = original_align
scorer.distance = official_score.distance


## Look at failures before the final run
The table shows the weakest validation sequences. Compare their images and transcripts;
separate detection misses, OCR errors, exclusions, ordering mistakes, and identity swaps.
Do not inspect or edit test predictions to tune the system.

In [ ]:
report = json.loads((WORK / f'{best.experiment}_scores.json').read_text())
worst = pd.DataFrame.from_dict(report['sequences'], orient='index')
display(worst[['text_order_score', 'balanced_joint_f1', 'matched_token_coverage']]
        .sort_values('balanced_joint_f1').head(8))
review_id = worst.sort_values('balanced_joint_f1').index[0]
predicted = scorer.load_jsonl(WORK / f'{best.experiment}.jsonl')[review_id]
for p in range(3):
    print(f'Page {p+1}')
    display(pd.DataFrame(predicted[p]))
    display(pd.DataFrame(gold[review_id][p]))
    display(Image.open(DATA_ROOT / seq_by_id[review_id]['images'][p]).resize((600, 850)))

## Refit and generate the test file
Refit the two small classifiers on all development sequences after selecting thresholds.
Saved validation results still refer to the earlier training-only models. Test images
are first processed here, and test references are never needed.

In [ ]:
final_heads, final_audit, final_counts = fit_heads(dev_ids)
joblib.dump({'heads': final_heads, 'config': selected, 'metadata': metadata},
            WORK / 'trained_pipeline.joblib')
final_audit.to_csv(WORK / 'final_alignment_audit.csv', index=False)
(WORK / 'final_training_counts.json').write_text(json.dumps(final_counts, indent=2))

def infer_test(bundle_path):
    bundle = joblib.load(bundle_path)
    assert bundle['metadata']['revision'] == revision
    predictions = {}
    for sid in tqdm(test_ids, desc='Test inference'):
        predictions[sid] = predict_pages(extract_sequence(sid), bundle['heads'], **bundle['config'])
    return predictions

test_predictions = infer_test(WORK / 'trained_pipeline.joblib')
submission = WORK / 'submission.jsonl'
write_jsonl(submission, test_predictions)
checked = scorer.load_jsonl(submission)
expected = scorer.load_jsonl(DATA_ROOT / 'sample_submission.jsonl')
assert set(checked) == set(expected) == set(test_ids)
assert len(checked) == 15
print('Validated:', submission)
print('Text entries:', sum(len(page) for pages in checked.values() for page in pages))

## Save the run
Download the archive and this executed notebook. The archive includes the trained small
models, split, model revision, dependency versions, alignment audits, metrics, and test
JSONL. Pretrained Magi weights are downloaded from the cited model repository rather
than bundled. Cached OCR/features stay in the notebook output for reruns.

Write your own short README after running: why this approach, the actual comparison
table, two concrete failures, what you changed, and what you would try next. Distinguish
the borrowed detector/OCR/embeddings from the trained heads and clustering harness.

Limitations: weak negative labels; no recovery of undetected balloons; fixed upstream
reading order; uncertain punctuation OCR; conservative identity merging; no learned
off-panel speaker resolution. This notebook does not implement the full joint graph
model discussed earlier.

Sources: [Magi code](https://github.com/ragavsachdeva/magi),
[MagiV2 weights and usage](https://huggingface.co/ragavsachdeva/magiv2),
[Tails Tell Tales](https://arxiv.org/abs/2408.00298),
[scikit-learn](https://scikit-learn.org/stable/).

In [ ]:
import zipfile
archive = WORK.parent / 'manga_tuned_notebook_artifacts.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in WORK.iterdir():
        if path.is_file():
            z.write(path, path.name)
from IPython.display import FileLink
print('Saved archive:', archive)
print(f'Total elapsed since setup: {(time.perf_counter()-RUN_STARTED)/60:.1f} minutes')